In [ ]:
%pip install dotenv

In [ ]:
import dotenv

dotenv.load_dotenv()

### Prmopt Template


In [ ]:
# source: https://freedium-mirror.cfd/https://generativeai.pub/stanford-just-killed-prompt-engineering-with-8-words-and-i-cant-believe-it-worked-8349d6524d2b
SYSTEM_INSTRUCTION = '''
You are a helpful assistant.
For each query, please generate a set of 2 possible responses, each within a separate <response> tag.
Responses should each include a <text> and a numeric <probability>.
Please sample at random from the tails of the distribution, such that the probability of each response is less than 0.10.
'''

QUERY_PROMPT = '''Write Python code for the following method:

def greatest_common_divisor(a: int, b: int) -> int:
    """ Return a greatest common divisor of two integers a and b
    >>> greatest_common_divisor(3, 5)
    1
    >>> greatest_common_divisor(25, 15)
    5
    """
'''

In [ ]:
# # source: https://freedium-mirror.cfd/https://generativeai.pub/stanford-just-killed-prompt-engineering-with-8-words-and-i-cant-believe-it-worked-8349d6524d2b
# SYSTEM_INSTRUCTION = """
# You are a helpful assistant.
# For each query, please generate a set of five possible responses, each within a separate para.
# Responses should each include a ANSWER and a numeric PROBABILITY.
# Please sample at random from the tails of the distribution, such that the probability of each response is less than 0.10.
# """

# QUERY_PROMPT = """
# Tell me a joke about Bangladesh in Bengali in hillrious way
# """

In [ ]:
import pandas as pd 

df = pd.DataFrame({
    "task_id": [],
    "model": [],
    "version": [],
    "query": [],
    "code": []
})

df.head()

### Groq

[Documentation](https://console.groq.com/docs/quickstart)


In [ ]:
# !pip install -U groq

In [ ]:
# !pip show groq httpx

In [ ]:
import os
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))
model = "openai/gpt-oss-120b"

chat_completion = client.chat.completions.create(
    messages=[
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": QUERY_PROMPT},
    ],
    model=model,
)

import re

response_text = chat_completion.choices[0].message.content

responses = re.findall(r'<response>(.*?)</response>', response_text, re.DOTALL)

for idx, resp in enumerate(responses[:2]):
    code_match = re.search(r'<text>(.*?)</text>', resp, re.DOTALL)
    if code_match:
        code = code_match.group(1).strip()
        if '```python' in code:
            start = code.find('```python') + 9
            end = code.find('```', start)
            code = code[start:end].strip()
        with open('llm_responses.txt', 'a') as f:
            f.write(f'Model: {model} v{idx+1}\n')
            f.write(f'Response: {code}\n')
            f.write(f'\nPrompt Token Count: {chat_completion.usage.prompt_tokens}\n')
            f.write(f'Completion Token Count: {chat_completion.usage.completion_tokens}\n')
            f.write(f'Reasoning Token Count: {chat_completion.usage.completion_tokens_details.reasoning_tokens}\n')
            f.write(f'\n>> Total Token Count: {chat_completion.usage.total_tokens}\n')
            f.write(f'\n\n=================\n')
        df = pd.concat([df, pd.DataFrame([{
            "task_id": 1,
            "model": model,
            "version": f"v{idx+1}",
            "query": QUERY_PROMPT,
            "code": code
        }])], ignore_index=True)

# print("Model: ", model)
# print("Response: ", chat_completion.choices[0].message.content)
# print("\nPrompt Token Count: ", chat_completion.usage.prompt_tokens)
# print("Completion Token Count: ", chat_completion.usage.completion_tokens)
# print(
#     "Reasoning Token Count: ",
#     chat_completion.usage.completion_tokens_details.reasoning_tokens,
# )

# print("\n>> Total Token Count: ", chat_completion.usage.total_tokens)

### AIsuite

[Blog](https://exploringartificialintelligence.substack.com/p/how-to-install-and-use-the-aisuite)


In [ ]:
# !pip install -U aisuite
# !pip install aisuite

In [ ]:
import os
import aisuite as ai

aiclient = ai.Client()

user_input = QUERY_PROMPT

groq_models = [
    "moonshotai/kimi-k2-instruct-0905",
    # "gemma-7b-it",
    # "openai/gpt-oss-120b",
]

for model in groq_models:
	print("model: ", model)

	response = aiclient.chat.completions.create(
		model=f"groq:{model}",
		messages=[
			{"role": "system", "content": SYSTEM_INSTRUCTION},
			{"role": "user", "content": user_input},
		],
		# stream=True
	)

	import re

	response_text = response.choices[0].message.content

	responses = re.findall(r'<response>(.*?)</response>', response_text, re.DOTALL)

	for idx, resp in enumerate(responses[:2]):
		code_match = re.search(r'<text>(.*?)</text>', resp, re.DOTALL)
		if code_match:
			code = code_match.group(1).strip()
			if '```python' in code:
				start = code.find('```python') + 9
				end = code.find('```', start)
				code = code[start:end].strip()
			with open('llm_responses.txt', 'a') as f:
				f.write(f'Model: {model} v{idx+1}\n')
				f.write(f'Response: {code}\n')
				f.write(f'\nPrompt Token Count: {response.usage.prompt_tokens}\n')
				f.write(f'Completion Token Count: {response.usage.completion_tokens}\n')
				f.write(f'Reasoning Token Count: {response.usage.completion_tokens_details.reasoning_tokens if response.usage.completion_tokens_details else "N/A"}\n')
				f.write(f'\n>> Total Token Count: {response.usage.total_tokens}\n')
				f.write(f'\n\n=================\n')
			df = pd.concat([df, pd.DataFrame([{
				"task_id": 1,
				"model": model,
				"version": f"v{idx+1}",
				"query": QUERY_PROMPT,
				"code": code
			}])], ignore_index=True)

	# print("Response: ", response.choices[0].message.content)
	# print("\nPrompt Token Count: ", response.usage.prompt_tokens)
	# print("Completion Token Count: ", response.usage.completion_tokens)
	# print("Reasoning Token Count: ",response.usage.completion_tokens_details.reasoning_tokens if response.usage.completion_tokens_details else "N/A" )

	# print("\n>> Total Token Count: ", response.usage.total_tokens)


### Cerebras

[Documentation](https://cloud.cerebras.ai/platform/org_j8rc4j3k5jjp448ycptp8htv/models)


In [ ]:
# !pip install cerebras-cloud-sdk

In [ ]:
import os
from cerebras.cloud.sdk import Cerebras

client = Cerebras(api_key=os.getenv("CEREBRAS_API_KEY"))
model = "llama-3.3-70b"

chat_completion = client.chat.completions.create(
    messages=[
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": QUERY_PROMPT},
    ],
    model=model,
    max_completion_tokens=4048,
    temperature=0.2,
    top_p=1,
    stream=False,
)

import re

response_text = chat_completion.choices[0].message.content

responses = re.findall(r'<response>(.*?)</response>', response_text, re.DOTALL)

for idx, resp in enumerate(responses[:2]):
    code_match = re.search(r'<text>(.*?)</text>', resp, re.DOTALL)
    if code_match:
        code = code_match.group(1).strip()
        if '```python' in code:
            start = code.find('```python') + 9
            end = code.find('```', start)
            code = code[start:end].strip()
        with open('llm_responses.txt', 'a') as f:
            f.write(f'Model: {model} v{idx+1}\n')
            f.write(f'Response: {code}\n')
            f.write(f'\nPrompt Token Count: {chat_completion.usage.prompt_tokens}\n')
            f.write(f'Completion Token Count: {chat_completion.usage.completion_tokens}\n')
            f.write(f'Reasoning Token Count: {chat_completion.usage.completion_tokens_details.reasoning_tokens}\n')
            f.write(f'\n>> Total Token Count: {chat_completion.usage.total_tokens}\n')
            f.write(f'\n\n=================\n')
        df = pd.concat([df, pd.DataFrame([{
            "task_id": 1,
            "model": model,
            "version": f"v{idx+1}",
            "query": QUERY_PROMPT,
            "code": code
        }])], ignore_index=True)

# print("Model: ", model)
# print("Response: ", chat_completion.choices[0].message.content)
# print("\nPrompt Token Count: ", chat_completion.usage.prompt_tokens)
# print("Completion Token Count: ", chat_completion.usage.completion_tokens)
# print(
#     "Reasoning Token Count: ",
#     chat_completion.usage.completion_tokens_details.reasoning_tokens,
# )

# print("\n>> Total Token Count: ", chat_completion.usage.total_tokens)

### Anthropic


In [ ]:
# %pip install boto3 -q

In [ ]:
import boto3
import json
import os


def generate_response(
    prompt: str,
    max_tokens: int = 4048,
    temperature: float = 0.1,
    track_tokens: bool = False,
):
    """Generate response from Claude via AWS Bedrock with optional token tracking."""

    client = boto3.client(
        "bedrock-runtime",
        region_name=os.getenv("AWS_DEFAULT_REGION", "us-east-1"),
        aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
        aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    )

    model_id = os.getenv("DEFAULT_MODEL", "us.anthropic.claude-sonnet-4-20250514-v1:0")
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "messages": [
            {"role": "user", "content": SYSTEM_INSTRUCTION+"\n\n"+prompt},
        ],
    }

    response = client.invoke_model(
        modelId=model_id, body=json.dumps(request_body), contentType="application/json"
    )

    response_body = json.loads(response["body"].read())
    text = ""
    if "content" in response_body and len(response_body["content"]) > 0:
        text = response_body["content"][0]["text"]

    if track_tokens:
        usage = response_body.get("usage", {})
        input_tokens = usage.get("input_tokens", 0)
        output_tokens = usage.get("output_tokens", 0)
        total_tokens = usage.get(
            "total_tokens",
            input_tokens + output_tokens,
        )
        return {
            "text": text,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
        }

    return text


anthropic_response = generate_response(QUERY_PROMPT, track_tokens=True)
# print(anthropic_response["text"])
# print(
#     f"Anthropic tokens → input={anthropic_response['input_tokens']}, "
#     f"output={anthropic_response['output_tokens']}, "
#     f"total={anthropic_response['total_tokens']}"
# )

import re

response_text = anthropic_response["text"]

responses = re.findall(r'<response>(.*?)</response>', response_text, re.DOTALL)

for idx, resp in enumerate(responses[:2]):
    code_match = re.search(r'<text>(.*?)</text>', resp, re.DOTALL)
    if code_match:
        code = code_match.group(1).strip()
        if '```python' in code:
            start = code.find('```python') + 9
            end = code.find('```', start)
            code = code[start:end].strip()
        with open('llm_responses.txt', 'a') as f:
            f.write(f'Model: claude-sonnet-4 v{idx+1}\n')
            f.write(f'Response: {code}\n')
            f.write(f'\nPrompt Token Count: {anthropic_response["input_tokens"]}\n')
            f.write(f'Completion Token Count: {anthropic_response["output_tokens"]}\n')
            f.write(f'Reasoning Token Count: {anthropic_response.get("reasoning_tokens", 0)}\n')
            f.write(f'\n>> Total Token Count: {anthropic_response["total_tokens"]}\n')
            f.write(f'\n\n=================\n')
        df = pd.concat([df, pd.DataFrame([{
            "task_id": 1,
            "model": "claude-sonnet-4",
            "version": f"v{idx+1}",
            "query": QUERY_PROMPT,
            "code": code
        }])], ignore_index=True)

# print("Model: claude-sonnet-4")
# print("Response: ", anthropic_response["text"])
# print("\nPrompt Token Count: ", anthropic_response['input_tokens'])
# print("Completion Token Count: ", anthropic_response['output_tokens'])
# print(
#     "Reasoning Token Count: ",
#     anthropic_response.get('reasoning_tokens', 0),
# )

# print("\n>> Total Token Count: ", anthropic_response['total_tokens'])

### Gemini

This section uses the shared system instruction and query to ask Gemini for the same mean_absolute_deviation implementation.

**Sanoke Query: "Give me the name of Google Free Medical LLMs Name only and their capability in 1 line"**

#### Free API access of GEMINI are provides by:

- `gemini-2.5-flash` (consumes too much thought tokens ~ 1600)
- `gemini-3-flash-preview` (consume too much due to thought tokens ~ 800) 🌟
- `gemini-flash-latest` (no thought tokens, totaltoken ~ 700 but incomplete results provided in some cases)
- `gemini-flash-lite-latest` (no thought tokens, totaltoken ~ 38 but result sometimes is wrong or incomplete)
- `gemini-2.5-flash-lite` (no thought tokens, totaltoken ~ 80)

**The core issue was google didn't provide Free API Access to their `pro` models. We can access only the `flash` or `lite` models for free.**


In [ ]:
# !pip install google-genai

In [ ]:
from google import genai
from google.genai import types
import os
import re
import pandas as pd

apikey = os.getenv("GEMINI_API_KEY")
model = "gemini-flash-latest"

client = genai.Client(
    api_key=apikey,
)

contents = [
    types.Content(
        role="user",
        parts=[types.Part.from_text(text=QUERY_PROMPT)],
    )
]

response = client.models.generate_content(
    model=model,
    contents=contents,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION
    ),
)

response_text = response.text

responses = re.findall(
    r'<response>(.*?)</response>',
    response_text,
    re.DOTALL
)

for idx, resp in enumerate(responses[:2]):
    code_match = re.search(
        r'<text>(.*?)</text>',
        resp,
        re.DOTALL
    )

    if code_match:
        code = code_match.group(1).strip()

        if '```python' in code:
            start = code.find('```python') + len('```python')
            end = code.find('```', start)
            code = code[start:end].strip()

        with open('llm_responses.txt', 'a') as f:
            f.write(f'Model: {model} v{idx+1}\n')
            f.write(f'Response: {code}\n')
            f.write(f'\nPrompt Token Count: {response.usage_metadata.prompt_token_count}\n')
            f.write(f'Completion Token Count: {response.usage_metadata.candidates_token_count}\n')
            f.write(f'\n>> Total Token Count: {response.usage_metadata.total_token_count}\n')
            f.write('\n=================\n')

        df = pd.concat(
            [
                df,
                pd.DataFrame([{
                    "task_id": 1,
                    "model": model,
                    "version": f"v{idx+1}",
                    "query": QUERY_PROMPT,
                    "code": code
                }])
            ],
            ignore_index=True
        )

In [ ]:
df.to_csv('generated_code_human_eval_1.csv', index=False)